# 8. Encoder-decoder LSTM 모델 해석

이 노트북은 `6_modeling.ipynb`에서 학습한 encoder-decoder multi-horizon LSTM을 해석합니다.

주요 출력:
- encoder-decoder LSTM의 permutation feature importance
- `t-3`, `t-2`, `t-1`, `t`별 permutation time-step importance
- `RUN_SHAP = True`일 때 SHAP feature importance

해석 대상은 원래의 encoder-decoder LSTM으로 제한하고, 새 single-output LSTM이나 ML/MLP baseline은 포함하지 않습니다.


In [ ]:
from pathlib import Path
import json
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import average_precision_score, roc_auc_score
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")


In [ ]:
# 작업 위치
PROJECT_DIR = Path.cwd().resolve().parent
DATA_SPLIT_DIR = PROJECT_DIR / "processed" / "data_split"
CLEAN_DATA_DIR = PROJECT_DIR / "models" / "clean_data"
MODEL_DIR = PROJECT_DIR / "models"
OUTPUT_DIR = PROJECT_DIR / "outputs" / "model_interpretation"
FIGURE_DIR = OUTPUT_DIR / "figures"

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_SPLIT_DIR:", DATA_SPLIT_DIR)
print("MODEL_DIR:", MODEL_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:
# 설정값 (config)
RANDOM_STATE = 42
HORIZONS = ["y_t", "y_t_plus_1", "y_t_plus_2"]
TIME_LABELS = ["t_minus_3", "t_minus_2", "t_minus_1", "t"]

MODEL_PATH = MODEL_DIR / "lstm_best_model_gpu.pt"
FEATURE_COLUMNS_PATH = CLEAN_DATA_DIR / "lstm_feature_columns.json"

EXPLAIN_SAMPLE_SIZE = 512
PERMUTATION_REPEATS = 3
TOP_N_PLOT = 25

RUN_SHAP = False
SHAP_BACKGROUND_SIZE = 128
SHAP_EXPLAIN_SIZE = 128
SHAP_HORIZON_INDEX = 2  # 0=y_t, 1=y_t_plus_1, 2=y_t_plus_2


In [ ]:
# device와 seed 설정
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("device:", device)
    print("gpu:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("device:", device)

PIN_MEMORY = device.type == "cuda"
NON_BLOCKING = device.type == "cuda"
AMP_ENABLED = device.type == "cuda"

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)


## 전처리 산출물 로딩


In [ ]:
required_files = {
    "X_train": DATA_SPLIT_DIR / "X_train_lstm.npy",
    "X_test": DATA_SPLIT_DIR / "X_test_lstm.npy",
    "y_train_steps": DATA_SPLIT_DIR / "y_train_steps_lstm.npy",
    "y_test_steps": DATA_SPLIT_DIR / "y_test_steps_lstm.npy",
    "y_train_step_mask": DATA_SPLIT_DIR / "y_train_step_mask_lstm.npy",
    "y_test_step_mask": DATA_SPLIT_DIR / "y_test_step_mask_lstm.npy",
    "meta_train": DATA_SPLIT_DIR / "lstm_train_metadata.csv",
    "meta_test": DATA_SPLIT_DIR / "lstm_test_metadata.csv",
}

X_train = np.load(required_files["X_train"]).astype(np.float32)
X_test = np.load(required_files["X_test"]).astype(np.float32)
y_train_steps = np.load(required_files["y_train_steps"]).astype(np.float32)
y_test_steps = np.load(required_files["y_test_steps"]).astype(np.float32)
y_train_step_mask = np.load(required_files["y_train_step_mask"]).astype(np.float32)
y_test_step_mask = np.load(required_files["y_test_step_mask"]).astype(np.float32)
meta_train = pd.read_csv(required_files["meta_train"])
meta_test = pd.read_csv(required_files["meta_test"])

with open(FEATURE_COLUMNS_PATH, "r", encoding="utf-8") as f:
    feature_columns = json.load(f)

print("X_train", X_train.shape)
print("X_test", X_test.shape)
print("features", len(feature_columns))


In [ ]:
data_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "n_sequences": X_train.shape[0],
            "sequence_length": X_train.shape[1],
            "n_features": X_train.shape[2],
            "masked_positive_rate_within_t_plus_2": float(((y_train_steps * y_train_step_mask).max(axis=1) > 0).mean()),
            "nan_count": int(np.isnan(X_train).sum()),
        },
        {
            "split": "test",
            "n_sequences": X_test.shape[0],
            "sequence_length": X_test.shape[1],
            "n_features": X_test.shape[2],
            "masked_positive_rate_within_t_plus_2": float(((y_test_steps * y_test_step_mask).max(axis=1) > 0).mean()),
            "nan_count": int(np.isnan(X_test).sum()),
        },
    ]
)
display(data_summary)


## Encoder-decoder LSTM checkpoint 로딩


In [ ]:
class LSTMEncoderDecoderClassifier(nn.Module):
    def __init__(self, input_size: int, hidden_size: int, num_layers: int, dropout: float, output_size: int = 3):
        super().__init__()
        lstm_dropout = dropout if num_layers > 1 else 0.0
        self.output_size = output_size
        self.encoder = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=lstm_dropout,
        )
        self.horizon_embedding = nn.Embedding(output_size, hidden_size)
        self.decoder = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=lstm_dropout,
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, 1)

    def forward(self, x):
        batch_size = x.size(0)
        _, (hidden, cell) = self.encoder(x)
        horizon_ids = torch.arange(self.output_size, device=x.device).unsqueeze(0).expand(batch_size, -1)
        decoder_input = self.horizon_embedding(horizon_ids)
        decoder_output, _ = self.decoder(decoder_input, (hidden, cell))
        return self.classifier(self.dropout(decoder_output)).squeeze(-1)


In [ ]:
model_payload = torch.load(MODEL_PATH, map_location=device, weights_only=False)

params = model_payload["params"]
model = LSTMEncoderDecoderClassifier(
    input_size=int(model_payload["input_size"]),
    hidden_size=params["hidden_size"],
    num_layers=params["num_layers"],
    dropout=params["dropout"],
    output_size=int(model_payload["output_size"]),
).to(device)
model.load_state_dict(model_payload["model_state_dict"])
model.eval()

print("loaded:", MODEL_PATH)
print("architecture:", model_payload["model_architecture"])
print("params:", params)


## 예측과 metric helper


In [ ]:
def make_predict_loader(x: np.ndarray, batch_size: int = 512) -> DataLoader:
    dummy_y = np.zeros((len(x), len(HORIZONS)), dtype=np.float32)
    dataset = TensorDataset(torch.from_numpy(x).float(), torch.from_numpy(dummy_y).float())
    return DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=PIN_MEMORY)


def predict_proba(model: nn.Module, x: np.ndarray, batch_size: int = 512) -> np.ndarray:
    model.eval()
    loader = make_predict_loader(x, batch_size=batch_size)
    probs = []
    with torch.no_grad():
        for xb, _ in loader:
            xb = xb.to(device, non_blocking=NON_BLOCKING)
            with torch.amp.autocast(device_type=device.type, enabled=AMP_ENABLED):
                logits = model(xb)
            probs.append(torch.sigmoid(logits.float()).detach().cpu().numpy())
    return np.concatenate(probs, axis=0)


def safe_auprc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    return float(average_precision_score(y_true.astype(int), y_prob)) if len(np.unique(y_true)) == 2 else np.nan


def safe_auroc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    return float(roc_auc_score(y_true.astype(int), y_prob)) if len(np.unique(y_true)) == 2 else np.nan


def masked_metric_summary(y_steps: np.ndarray, y_prob: np.ndarray, y_mask: np.ndarray) -> dict:
    y_prob_masked = np.where(y_mask.astype(bool), y_prob, 0.0)
    summary = {}
    horizon_auprcs = []
    horizon_aurocs = []
    for idx, horizon in enumerate(HORIZONS):
        active = y_mask[:, idx].astype(bool)
        y_true_h = y_steps[active, idx]
        y_prob_h = y_prob[active, idx]
        auprc = safe_auprc(y_true_h, y_prob_h)
        auroc = safe_auroc(y_true_h, y_prob_h)
        summary[f"{horizon}_auprc"] = auprc
        summary[f"{horizon}_auroc"] = auroc
        summary[f"{horizon}_n"] = int(active.sum())
        horizon_auprcs.append(auprc)
        horizon_aurocs.append(auroc)

    y_true_within = ((y_steps * y_mask).max(axis=1) > 0).astype(int)
    y_prob_within = 1.0 - np.prod(1.0 - y_prob_masked, axis=1)
    summary["macro_auprc"] = float(np.nanmean(horizon_auprcs))
    summary["macro_auroc"] = float(np.nanmean(horizon_aurocs))
    summary["within_t_plus_2_auprc"] = safe_auprc(y_true_within, y_prob_within)
    summary["within_t_plus_2_auroc"] = safe_auroc(y_true_within, y_prob_within)
    return summary


In [ ]:
test_prob = predict_proba(model, X_test)
test_summary = masked_metric_summary(y_test_steps, test_prob, y_test_step_mask)
display(pd.DataFrame([test_summary]))


## 해석 대상 subset 선택


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)

X_explain_source = X_test
y_explain_steps = y_test_steps
y_explain_mask = y_test_step_mask
meta_explain = meta_test

n_explain = min(EXPLAIN_SAMPLE_SIZE, len(X_explain_source))
explain_idx = rng.choice(len(X_explain_source), size=n_explain, replace=False)
X_explain = X_explain_source[explain_idx].copy()
y_explain_steps_sub = y_explain_steps[explain_idx].copy()
y_explain_mask_sub = y_explain_mask[explain_idx].copy()
meta_explain_sub = meta_explain.iloc[explain_idx].reset_index(drop=True)

baseline_prob = predict_proba(model, X_explain)
baseline_metrics = masked_metric_summary(y_explain_steps_sub, baseline_prob, y_explain_mask_sub)
print("explain subset:", X_explain.shape)
display(pd.DataFrame([baseline_metrics]))


## Permutation feature importance

각 feature에 대해 `t-3`부터 `t`까지의 전체 trajectory를 sequence 사이에서 섞습니다. 이렇게 하면 해당 feature의 4-step 패턴은 유지하면서 target과의 관계만 끊어 중요도를 추정할 수 있습니다.


In [ ]:
def permutation_feature_importance(
    model: nn.Module,
    x: np.ndarray,
    y_steps: np.ndarray,
    y_mask: np.ndarray,
    feature_names: list[str],
    repeats: int,
) -> pd.DataFrame:
    baseline_prob = predict_proba(model, x)
    baseline = masked_metric_summary(y_steps, baseline_prob, y_mask)
    feature_indices = list(range(x.shape[2]))

    rows = []
    for feature_idx in feature_indices:
        repeat_rows = []
        for repeat in range(1, repeats + 1):
            x_perm = x.copy()
            perm = rng.permutation(x_perm.shape[0])
            x_perm[:, :, feature_idx] = x_perm[perm, :, feature_idx]
            perm_prob = predict_proba(model, x_perm)
            metrics = masked_metric_summary(y_steps, perm_prob, y_mask)
            repeat_rows.append(metrics)

        repeat_df = pd.DataFrame(repeat_rows)
        row = {
            "feature_idx": feature_idx,
            "feature": feature_names[feature_idx],
            "repeats": repeats,
        }
        for metric_name, baseline_value in baseline.items():
            if metric_name.endswith("_n"):
                continue
            perm_mean = float(repeat_df[metric_name].mean())
            perm_std = float(repeat_df[metric_name].std(ddof=0))
            row[f"baseline_{metric_name}"] = baseline_value
            row[f"permuted_mean_{metric_name}"] = perm_mean
            row[f"permuted_std_{metric_name}"] = perm_std
            row[f"drop_{metric_name}"] = baseline_value - perm_mean
        rows.append(row)

    return pd.DataFrame(rows)


feature_importance = permutation_feature_importance(
    model=model,
    x=X_explain,
    y_steps=y_explain_steps_sub,
    y_mask=y_explain_mask_sub,
    feature_names=feature_columns,
    repeats=PERMUTATION_REPEATS,
)
feature_importance = feature_importance.sort_values("drop_within_t_plus_2_auprc", ascending=False).reset_index(drop=True)
feature_importance.to_csv(OUTPUT_DIR / "encoder_decoder_lstm_permutation_feature_importance.csv", index=False)
display(feature_importance.head(30))


In [ ]:
plot_df = feature_importance.head(TOP_N_PLOT).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, max(5, 0.28 * len(plot_df))))
ax.barh(plot_df["feature"], plot_df["drop_within_t_plus_2_auprc"], color="tab:blue")
ax.set_xlabel("AUPRC drop after permutation")
ax.set_ylabel("Feature")
ax.set_title("Encoder-decoder LSTM permutation feature importance")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "encoder_decoder_lstm_permutation_feature_importance_top.png", dpi=200, bbox_inches="tight")
plt.show()


## Permutation time-step importance

각 time step을 sequence 사이에서 섞되, 해당 time step 안의 전체 feature vector는 유지합니다.


In [ ]:
def permutation_time_importance(
    model: nn.Module,
    x: np.ndarray,
    y_steps: np.ndarray,
    y_mask: np.ndarray,
    repeats: int,
) -> pd.DataFrame:
    baseline_prob = predict_proba(model, x)
    baseline = masked_metric_summary(y_steps, baseline_prob, y_mask)
    rows = []
    for time_idx, time_label in enumerate(TIME_LABELS):
        repeat_rows = []
        for repeat in range(1, repeats + 1):
            x_perm = x.copy()
            perm = rng.permutation(x_perm.shape[0])
            x_perm[:, time_idx, :] = x_perm[perm, time_idx, :]
            perm_prob = predict_proba(model, x_perm)
            repeat_rows.append(masked_metric_summary(y_steps, perm_prob, y_mask))

        repeat_df = pd.DataFrame(repeat_rows)
        row = {"time_idx": time_idx, "time_label": time_label, "repeats": repeats}
        for metric_name, baseline_value in baseline.items():
            if metric_name.endswith("_n"):
                continue
            perm_mean = float(repeat_df[metric_name].mean())
            row[f"baseline_{metric_name}"] = baseline_value
            row[f"permuted_mean_{metric_name}"] = perm_mean
            row[f"drop_{metric_name}"] = baseline_value - perm_mean
        rows.append(row)
    return pd.DataFrame(rows)


time_importance = permutation_time_importance(model, X_explain, y_explain_steps_sub, y_explain_mask_sub, PERMUTATION_REPEATS)
time_importance = time_importance.sort_values("drop_within_t_plus_2_auprc", ascending=False).reset_index(drop=True)
time_importance.to_csv(OUTPUT_DIR / "encoder_decoder_lstm_permutation_time_importance.csv", index=False)
display(time_importance)


In [ ]:
plot_df = time_importance.sort_values("time_idx")
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(plot_df["time_label"], plot_df["drop_within_t_plus_2_auprc"], color="tab:green")
ax.set_xlabel("Input time step")
ax.set_ylabel("AUPRC drop after permutation")
ax.set_title("Encoder-decoder LSTM time-step importance")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "encoder_decoder_lstm_permutation_time_importance.png", dpi=200, bbox_inches="tight")
plt.show()


## SHAP

`RUN_SHAP = True`로 설정한 경우 선택한 horizon logit 하나를 설명하고, absolute SHAP value를 sample과 time step 방향으로 집계합니다.


In [ ]:
print("RUN_SHAP:", RUN_SHAP)

if RUN_SHAP:
    import shap

    class HorizonLogitWrapper(nn.Module):
        def __init__(self, base_model: nn.Module, horizon_index: int):
            super().__init__()
            self.base_model = base_model
            self.horizon_index = horizon_index

        def forward(self, x):
            return self.base_model(x)[:, self.horizon_index]

    bg_n = min(SHAP_BACKGROUND_SIZE, len(X_train))
    ex_n = min(SHAP_EXPLAIN_SIZE, len(X_explain))
    bg_idx = rng.choice(len(X_train), size=bg_n, replace=False)
    shap_x = torch.from_numpy(X_explain[:ex_n]).float().to(device)
    shap_background = torch.from_numpy(X_train[bg_idx]).float().to(device)

    shap_model = HorizonLogitWrapper(model, SHAP_HORIZON_INDEX).to(device).eval()
    explainer = shap.DeepExplainer(shap_model, shap_background)
    shap_values = explainer.shap_values(shap_x)

    if isinstance(shap_values, list):
        shap_values = shap_values[0]
    shap_values = np.asarray(shap_values)
    if shap_values.ndim == 4 and shap_values.shape[-1] == 1:
        shap_values = shap_values[..., 0]

    mean_abs_by_feature = np.abs(shap_values).mean(axis=(0, 1))
    shap_feature_importance = pd.DataFrame(
        {
            "feature_idx": np.arange(len(feature_columns)),
            "feature": feature_columns,
            "mean_abs_shap": mean_abs_by_feature,
            "horizon": HORIZONS[SHAP_HORIZON_INDEX],
        }
    ).sort_values("mean_abs_shap", ascending=False)
    shap_feature_importance.to_csv(OUTPUT_DIR / "encoder_decoder_lstm_shap_feature_importance.csv", index=False)
    display(shap_feature_importance.head(30))

    plot_df = shap_feature_importance.head(TOP_N_PLOT).iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, max(5, 0.28 * len(plot_df))))
    ax.barh(plot_df["feature"], plot_df["mean_abs_shap"], color="tab:purple")
    ax.set_xlabel("Mean |SHAP value|")
    ax.set_ylabel("Feature")
    ax.set_title(f"Encoder-decoder LSTM SHAP feature importance ({HORIZONS[SHAP_HORIZON_INDEX]})")
    ax.grid(axis="x", alpha=0.3)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "encoder_decoder_lstm_shap_feature_importance_top.png", dpi=200, bbox_inches="tight")
    plt.show()


## 저장된 산출물


In [ ]:
saved_files = sorted([str(path.relative_to(PROJECT_DIR)) for path in OUTPUT_DIR.glob("*.csv")])
saved_figures = sorted([str(path.relative_to(PROJECT_DIR)) for path in FIGURE_DIR.glob("*.png")])
print("CSV outputs:")
for path in saved_files:
    print("-", path)
print("Figure outputs:")
for path in saved_figures:
    print("-", path)
